# MoE routing experiment v2: your router vs two baselines

**What changed from v1** (the fixes from the 29 Sep review):
1. Quality is measured on **held-out text**: the last 10% of the text, which is never used for training.
2. Your router now works with **prices**, and all tokens choose at once. Each expert has a *load price* that rises when it's busy. Travel has its own price (β) that rises automatically whenever communication goes above a budget you set. This fixes the drift problem, and it's fast.
3. A **stronger baseline**: DeepSeek-style aux-free balancing (load prices, no travel cost).
4. **Fair pairing**: for the same seed, every router sees the same data in the same order and starts from the same weights.
5. **5 seeds, longer training, 2 MoE layers**, a plot with error bars, and a **confidence interval** that says whether quality really stayed the same.
6. Communication is also counted the way real systems send data: each destination GPU once per token ("dedup").

**How to run**
- **Kaggle:** Settings → Accelerator → GPU T4 or P100, and Settings → Internet → On.
- **Colab:** Runtime → Change runtime type → T4 GPU.
- **Mac:** set `QUICK = True` in the settings cell.

Then **Run all**.

**Time:** the full run is 25 training runs, about 48 seconds each on a Kaggle T4. `QUICK = True` takes a few minutes.

In [ ]:
import math, time, urllib.request, statistics as st
import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    DEVICE = "cuda"          # NVIDIA GPU (Colab / Kaggle)
elif torch.backends.mps.is_available():
    DEVICE = "mps"           # Apple GPU (Mac)
else:
    DEVICE = "cpu"
print("device:", DEVICE)

## Settings
You only ever need to touch this cell. `COMM_TARGETS` are the communication budgets to try. A router that ignores location sits at about 0.575, so a budget of 0.35 means "cut communication by roughly 40%".

In [ ]:
QUICK = False                  # True = small fast run (Mac/CPU or a quick check)

SEEDS = [1, 2, 3, 4, 5]        # same seed = same data order and same starting weights for every router
STEPS = 2000                   # training steps per run
BS, BLOCK = 32, 64             # 32 sequences of 64 characters per batch
D_MODEL, N_LAYERS, HEADS, HIDDEN = 128, 2, 4, 256
LR = 2e-3

E, K = 8, 2                    # 8 experts per MoE layer, each token uses 2

AUX_COEF = 0.01                # weight of the aux balance loss (baseline 1)
ETA_P = 0.05                   # how fast load prices move
ETA_BETA = 0.1                 # how fast the travel price (beta) moves
COMM_TARGETS = [0.45, 0.35, 0.25]
EVAL_BATCHES = 50              # held-out batches used for the final score

if QUICK:
    SEEDS, STEPS, D_MODEL, HIDDEN = [1, 2, 3], 400, 64, 128
    COMM_TARGETS, EVAL_BATCHES = [0.4, 0.3], 20
print(f"{len(SEEDS)} seeds x {2 + len(COMM_TARGETS)} routers = {len(SEEDS) * (2 + len(COMM_TARGETS))} runs of {STEPS} steps")

## 1. Data
Tiny Shakespeare at character level: the model reads 64 characters and predicts the next one. The first 90% is for training. The last 10% is held out and only used to score the model.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    with urllib.request.urlopen(URL, timeout=30) as r:
        text = r.read().decode("utf-8")
except Exception as err:
    raise RuntimeError("Could not download the text. On Kaggle, turn on Settings -> Internet.") from err

chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"{len(text):,} characters | vocab {V} | train {len(train_data):,} | held-out {len(val_data):,}")

def get_batch(src, gen):
    ix = torch.randint(0, len(src) - BLOCK - 1, (BS,), generator=gen)
    x = torch.stack([src[i:i + BLOCK] for i in ix])
    y = torch.stack([src[i + 1:i + BLOCK + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

## 2. Pretend GPUs and travel cost
The 8 experts are spread over 4 pretend GPUs on 2 pretend machines, and each sequence sits on one pretend GPU. The travel cost `D` is 0 for the same GPU, 0.3 for the same machine and 1.0 for a different machine.

In [ ]:
N_GPU, GPU_PER_NODE = 4, 2
SAME_NODE, CROSS_NODE = 0.3, 1.0
D = torch.zeros(N_GPU, N_GPU)
for i in range(N_GPU):
    for j in range(N_GPU):
        if i != j:
            D[i, j] = SAME_NODE if i // GPU_PER_NODE == j // GPU_PER_NODE else CROSS_NODE
D = D.to(DEVICE)
EXP_GPU = (torch.arange(E) % N_GPU).to(DEVICE)     # expert e lives on pretend GPU e % 4
print(D.cpu())
print(f"comm of a router that ignores location: about {D.mean().item():.3f}")

## 3. Model and the three routers
- **top-k + aux loss** (baseline 1): pick the 2 highest gate scores; an extra training loss discourages overload.
- **aux-free prices** (baseline 2, DeepSeek-V3 style): pick the top 2 of `gate − price_e`. An overloaded expert's price goes up after each step.
- **ours**: pick the top 2 of `gate − β·travel_cost − price_e`. β goes up whenever communication is above the budget and down when it's below. This is your derived rule, with both prices learned by *dual ascent*.

In all three, the mixing weights come from the plain gate, so only *which* experts get picked differs.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Linear(h, d))
    def forward(self, x):
        return self.net(x)


class MoELayer(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.gate = nn.Linear(d, E)
        self.experts = nn.ModuleList([Expert(d, h) for _ in range(E)])
        self.register_buffer("price", torch.zeros(E))    # load price per expert
        self.register_buffer("beta", torch.zeros(()))    # travel (communication) price

    def forward(self, x, router, tok_gpu):
        B, T, Dm = x.shape
        flat = x.reshape(-1, Dm)
        N = flat.shape[0]
        logits = self.gate(flat)
        probs = F.softmax(logits, dim=-1)
        tg = tok_gpu.repeat_interleave(T)                # pretend GPU of every token
        cost = D[tg][:, EXP_GPU]                          # N x E travel cost

        with torch.no_grad():                             # the choice itself is not trained directly
            if router == "topk_aux":
                score = logits
            elif router == "auxfree":
                score = logits - self.price
            elif router == "ours":
                score = logits - self.beta * cost - self.price
            else:
                raise ValueError(router)
            top_i = torch.topk(score, K, dim=-1).indices      # N x K chosen experts

        aux = torch.zeros((), device=x.device)
        if router == "topk_aux":
            frac = F.one_hot(top_i[:, 0], E).float().mean(0)
            aux = E * (frac * probs.mean(0)).sum()

        w = probs.gather(1, top_i)
        w = w / w.sum(-1, keepdim=True)
        rep = flat.repeat_interleave(K, dim=0)            # row n*K + s = token n, slot s
        idx = top_i.reshape(-1)
        out = torch.zeros_like(rep)
        for e in range(E):
            m = idx == e
            if m.any():
                out[m] = self.experts[e](rep[m])
        out = (out * w.reshape(-1, 1)).view(N, K, Dm).sum(1)

        with torch.no_grad():
            n = F.one_hot(idx, E).sum(0).float()          # tokens per expert
            comm = cost.gather(1, top_i).mean()
            dest, _ = torch.sort(EXP_GPU[top_i], dim=1)
            first = torch.ones_like(dest, dtype=torch.bool)
            first[:, 1:] = dest[:, 1:] != dest[:, :-1]    # count each destination GPU once
            comm_dedup = (D[tg[:, None], dest] * first).sum(1).mean() / K
        return out.reshape(B, T, Dm), aux, {"n": n, "comm": comm, "comm_dedup": comm_dedup}


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, HEADS, batch_first=True)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.moe = MoELayer(D_MODEL, HIDDEN)

    def forward(self, x, router, tok_gpu, mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        m, aux, stats = self.moe(self.ln2(x), router, tok_gpu)
        return x + m, aux, stats


class TinyMoEGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(V, D_MODEL)
        self.pos = nn.Embedding(BLOCK, D_MODEL)
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYERS)])
        self.ln = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, V)
        self.register_buffer("mask", torch.triu(torch.full((BLOCK, BLOCK), float("-inf")), 1))

    def forward(self, x, router, tok_gpu):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        aux_total, all_stats = 0.0, []
        for blk in self.blocks:
            h, aux, stats = blk(h, router, tok_gpu, self.mask[:T, :T])
            aux_total = aux_total + aux
            all_stats.append(stats)
        return self.head(self.ln(h)), aux_total, all_stats

## 4. Train one model, then score it on held-out text
After every training step, the prices are updated. That is the *dual ascent* step:
- `price_e += ETA_P × (load_e / average load − 1)`: busy experts get more expensive.
- `β += ETA_BETA × (comm − budget)`: if travel is above budget, travel gets more expensive (never below 0).

During scoring the prices are frozen, and every run is scored on the same held-out batches.

In [ ]:
def run(router, seed, target=None):
    torch.manual_seed(seed)                              # same starting weights for every router
    gen = torch.Generator().manual_seed(1000 + seed)     # same training data order for every router
    model = TinyMoEGPT().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    moes = [blk.moe for blk in model.blocks]
    t0 = time.time()
    model.train()
    for step in range(STEPS):
        x, y = get_batch(train_data, gen)
        tok_gpu = torch.randint(0, N_GPU, (BS,), generator=gen).to(DEVICE)
        logits, aux, stats = model(x, router, tok_gpu)
        loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        if router == "topk_aux":
            loss = loss + AUX_COEF * aux
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        with torch.no_grad():
            for moe, s in zip(moes, stats):
                if router in ("auxfree", "ours"):
                    moe.price += ETA_P * (s["n"] / s["n"].mean() - 1.0)
                if router == "ours":
                    moe.beta.add_(ETA_BETA * (s["comm"] - target)).clamp_(min=0.0)
    train_s = time.time() - t0

    model.eval()
    vgen = torch.Generator().manual_seed(7)
    L, C, CD, LD = [], [], [], []
    with torch.no_grad():
        for _ in range(EVAL_BATCHES):
            x, y = get_batch(val_data, vgen)
            tok_gpu = torch.randint(0, N_GPU, (BS,), generator=vgen).to(DEVICE)
            logits, _, stats = model(x, router, tok_gpu)
            L.append(F.cross_entropy(logits.reshape(-1, V), y.reshape(-1)).item())
            C.append(sum(s["comm"].item() for s in stats) / len(stats))
            CD.append(sum(s["comm_dedup"].item() for s in stats) / len(stats))
            LD.append(sum((s["n"].max() / s["n"].mean()).item() for s in stats) / len(stats))
    mean = lambda v: sum(v) / len(v)
    return {"ppl": math.exp(mean(L)), "comm": mean(C), "comm_dedup": mean(CD), "load": mean(LD),
            "beta": mean([m.beta.item() for m in moes]), "train_s": train_s}

## 5. Run everything
For every seed this trains the two baselines, then your router once per budget. Progress prints as it goes.

In [ ]:
plan = [("topk_aux", None), ("auxfree", None)] + [("ours", t) for t in COMM_TARGETS]
NAMES = {("topk_aux", None): "top-k + aux loss", ("auxfree", None): "aux-free prices (DeepSeek-style)"}
name_of = lambda router, target: NAMES.get((router, target), f"ours, comm budget {target}")

results = []
t_all = time.time()
for seed in SEEDS:
    for router, target in plan:
        r = run(router, seed, target)
        r.update(router=router, target=target, seed=seed)
        results.append(r)
        print(f"seed {seed} | {name_of(router, target):<33} | ppl {r['ppl']:.3f} | comm {r['comm']:.3f} "
              f"| load {r['load']:.2f} | beta {r['beta']:.2f} | {r['train_s']:.0f}s")
print(f"total time: {(time.time() - t_all) / 60:.1f} min")

## 6. Results table
Mean ± spread over seeds. For comm and load, lower is better. Load is the busiest expert divided by the average expert, so 1.00 means perfectly even.

In [ ]:
summary = {}
print(f"{'router':<34} {'held-out ppl':>15} {'comm':>13} {'comm (dedup)':>14} {'load':>12}")
for router, target in plan:
    rs = [r for r in results if r["router"] == router and r["target"] == target]
    s = {k: (st.mean([r[k] for r in rs]), st.stdev([r[k] for r in rs]) if len(rs) > 1 else 0.0)
         for k in ["ppl", "comm", "comm_dedup", "load", "beta"]}
    summary[(router, target)] = s
    pm = lambda k, p=3: f"{s[k][0]:.{p}f} ± {s[k][1]:.{p}f}"
    print(f"{name_of(router, target):<34} {pm('ppl'):>15} {pm('comm'):>13} {pm('comm_dedup'):>14} {pm('load', 2):>12}")

## 7. Did quality really stay the same?
This compares each router with **top-k + aux loss** seed by seed. Pairing is fair because each seed shares the same data and starting weights. It prints the average perplexity change and a 95% confidence interval.
- The whole interval inside ±1% → **same quality within 1%**.
- The whole interval above 0 → **worse**. The whole interval below 0 → **better**.
- Otherwise → **not sure yet** (it needs more seeds).

In [ ]:
try:
    from scipy import stats as sps
    tcrit = lambda n: sps.t.ppf(0.975, n - 1)
except ImportError:
    tcrit = lambda n: {2: 12.71, 3: 4.30, 4: 3.18, 5: 2.78, 6: 2.57, 7: 2.45, 8: 2.36, 9: 2.31, 10: 2.26}.get(n, 2.0)

base = {r["seed"]: r["ppl"] for r in results if r["router"] == "topk_aux"}
print("perplexity change vs top-k + aux loss (negative = better):")
for router, target in plan[1:]:
    d = [100 * (r["ppl"] / base[r["seed"]] - 1) for r in results if r["router"] == router and r["target"] == target]
    m = st.mean(d)
    half = tcrit(len(d)) * (st.stdev(d) if len(d) > 1 else 0.0) / math.sqrt(len(d))
    lo, hi = m - half, m + half
    if lo > -1 and hi < 1:
        verdict = "same quality within 1%" + (" (slightly worse)" if lo > 0 else " (slightly better)" if hi < 0 else "")
    elif lo > 0:
        verdict = "worse"
    elif hi < 0:
        verdict = "better"
    else:
        verdict = "not sure yet (more seeds needed)"
    print(f"  {name_of(router, target):<34} {m:+.2f}%   95% CI [{lo:+.2f}%, {hi:+.2f}%]   -> {verdict}")

## 8. The picture
Each point is a router, with error bars showing the spread over seeds. Good means **down** (better quality) and **left** (less communication).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for key, marker in [(("topk_aux", None), "o"), (("auxfree", None), "^")]:
    s = summary[key]
    ax.errorbar(s["comm"][0], s["ppl"][0], xerr=s["comm"][1], yerr=s["ppl"][1],
                marker=marker, ms=8, capsize=3, ls="none", label=name_of(*key))
ours = [summary[("ours", t)] for t in COMM_TARGETS]
ax.errorbar([s["comm"][0] for s in ours], [s["ppl"][0] for s in ours],
            xerr=[s["comm"][1] for s in ours], yerr=[s["ppl"][1] for s in ours],
            marker="s", ms=7, capsize=3, label="ours (one point per budget)")
for t, s in zip(COMM_TARGETS, ours):
    ax.annotate(f"budget {t}", (s["comm"][0], s["ppl"][0]), textcoords="offset points", xytext=(6, 6), fontsize=8)
ax.set_xlabel("communication cost (lower = better)")
ax.set_ylabel("held-out perplexity (lower = better)")
ax.set_title("Quality vs communication, mean ± spread over seeds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()